# Stage 1 — Reconstructing SegRNN's Table II and Table III
### ETTh1, ETTh2, ETTm1, ETTm2 — multivariate and univariate

## The problem this paper addresses

Long-term time-series forecasting (LTSF) means predicting many steps
into the future — horizons of H=96 up to H=720 timesteps — from a long
history of past observations (here, a look-back window of L=720). This
is hard for two different families of models, for two different
reasons.

**Classical statistical models** (ARIMA, exponential smoothing, and the
other models covered in this course's Time-Series Forecasting lecture)
are built around short-term, largely univariate structure — they assume
relatively simple, stable autocorrelation patterns and don't scale well
to long horizons or many correlated input channels (ETTh1 alone has 7).

**Recurrent neural networks** (RNNs/LSTMs/GRUs) can in principle model
arbitrary long-range dependencies, but in practice degrade badly on long
sequences, for two compounding reasons: (1) processing a 720-step
look-back one timestep at a time means backpropagating through 720
sequential steps, where gradients vanish or explode; and (2) generating
a long forecast one step at a time (autoregressively) means every
prediction feeds into the next one, so small errors compound over the
horizon — by H=720 the model is forecasting from its own accumulated
mistakes. This is why, before this paper, Transformer-based
architectures (Informer, Autoformer, FEDformer, PatchTST, iTransformer)
had become the dominant approach for LTSF: attention looks at the whole
sequence at once, sidestepping both problems — at the cost of being
computationally expensive (attention scales roughly quadratically with
sequence length) and having far more parameters to train.

The paper's question is direct: **can an RNN be redesigned to avoid
both of its usual failure modes, while staying much cheaper than a
Transformer?**

## How SegRNN solves it

SegRNN's answer is two structural changes to a plain GRU, neither of
which adds a new mechanism (no attention, no extra layers) — they
change *what the RNN operates over*:

**1. Segment-wise iteration, not point-wise iteration.** Instead of
feeding the GRU one raw timestep at a time (720 steps for L=720), the
look-back window is first chopped into `n = L/w` non-overlapping
segments of length `w` (24 or 48 here, chosen per dataset), each segment
is linearly embedded into a single d-dimensional vector, and *that*
sequence of `n` segment-vectors (e.g. n=15 or n=30, not 720) is what the
GRU actually encodes. This directly attacks the vanishing-gradient/
long-sequence problem: the GRU only has to propagate information across
a few dozen steps, not hundreds.

**2. Parallel Multi-step Forecasting (PMF), not step-by-step decoding.**
Instead of generating the forecast one segment at a time and feeding
each prediction back in as the next input (the classical,
error-compounding approach — the paper calls this RMF and shows it's
worse), SegRNN generates *all* `m` future segments in a single parallel
pass: each target segment gets its own positional embedding (its
relative position in the forecast horizon, concatenated with which
channel it belongs to), and all `m` of these embeddings are pushed
through the *same* GRU cell — the one already used for encoding — at
once, seeded with the encoder's final hidden state. No prediction ever
feeds into another prediction, so there's no error-accumulation chain
to begin with.

The result, per the paper's own numbers (reproduced below): a
single-layer GRU, doing almost nothing architecturally exotic, matches
or beats Transformer-based SOTA models on most settings, while using a
small fraction of the parameters (the paper's Table VI: >78% less
training time, >82% less peak GPU memory vs. PatchTST). That efficiency
claim is exactly what this project's Stage 2 work has been probing from
many angles (the `d_model` sweep, weight tying, the frozen-reservoir
experiment) — this notebook is the foundation those experiments build
on: reproducing the paper's own accuracy claims, faithfully, before
changing anything.

## What Table II and Table III each check, and why both

**Table II (multivariate)** is the paper's *main* result: all 7 input
channels are used both as input and as forecast target simultaneously,
and SegRNN is compared against 8 baselines across 8 datasets. This is
the setting the paper's headline claims are about, and the one this
project's Stage 2 experiments have all been built on (ETTh1,
multivariate).

**Table III (univariate)** asks a narrower but important question: does
the architecture still hold up in the simpler single-channel setting,
where there's no cross-channel information to exploit? The paper
disables the CP (channel-identity) half of the positional embedding for
this setting, since there's only one channel to identify — everything
else about the architecture is unchanged. Reconstructing both tells us
whether SegRNN's advantage is really about the segment-wise/PMF
mechanism itself (which should hold in both settings) or partly an
artifact of how it handles multiple correlated channels (which would
only show up in Table II).

This notebook reconstructs both tables for the four ETT datasets
(ETTh1, ETTh2, ETTm1, ETTm2) — the two hourly and two 15-minute variants
of the same underlying transformer-load sensor data, now uploaded to
Drive.

## Evaluation protocol (same for every run in this notebook)

- Look-back `L=720`, forecast horizons `H ∈ {96, 192, 336, 720}` — fixed
  by the paper, not tuned per dataset.
- Chronological 6:2:2 train/val/test split (12/4/4 months —
  `data_provider/data_loader.py`'s `Dataset_ETT_hour` for ETTh1/ETTh2,
  `Dataset_ETT_minute` for ETTm1/ETTm2 using the identical border
  formula scaled x4 for 15-minute-frequency data), `StandardScaler` fit
  on the train split only — the same protocol every Stage 2 strand in
  this project has used.
- Metrics: **MSE** and **MAE**, computed on the standardized (scaled)
  values, matching the paper's own evaluation, not inverse-transformed
  back to physical units. One small note on the paper's own tables: the
  second metric column is literally labeled "MSA" in the published PDF
  (`docs/SegRNN_paper.pdf`, Tables II and III), not "MAE" — the values
  match standard MAE exactly (e.g. ETTh1 H=96 multivariate: 0.392,
  consistent with every other citation of this result throughout this
  project), so this reads as a labeling artifact in the published table
  rather than a different metric — worth flagging as a genuine, checkable
  detail rather than glossing over it.
- Every run below uses the *exact* hyperparameters from this repo's own
  `scripts/SegRNN/<dataset>.sh` and `scripts/SegRNN/univariate.sh` —
  these differ meaningfully per dataset (segment length, dropout, batch
  size, learning rate, and whether channel-identity encoding is even
  used). The paper doesn't claim one universal hyperparameter setting;
  it tunes per dataset, and this notebook reproduces that faithfully
  rather than picking one convenient configuration.

**~32 training runs total** (4 datasets x 4 horizons x 2 settings
[multivariate/univariate]) — roughly 2-2.5 hours on a T4, similar per-run
cost to the first Stage 2 notebook. Each dataset/setting combination is
its own Part below, so a Colab disconnect only costs that one part, not
the whole run — restart from Setup and re-run only the Parts you haven't
completed yet.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
import os
REPO = "https://github.com/amitzr/SegRNN.git"
if not os.path.exists('/content/proj'):
    !git clone $REPO /content/proj
%cd /content/proj
!git pull

In [ ]:
import os
if not os.path.exists('/content/proj/dataset'):
    os.symlink('/content/drive/MyDrive/ts-project/dataset', '/content/proj/dataset')
!ls -la /content/proj/dataset | head
!pip install -q -r requirements.txt

## Setup: shared constants, training runner, plotting

`run_horizon` here is more general than the one in the Stage 2
notebooks — it's parameterized by dataset name, feature mode
(multivariate `M` / univariate `S`), and every hyperparameter that
varies per dataset, instead of hardcoding ETTh1's own values.

In [ ]:
import os, sys, re, csv, subprocess, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, os.getcwd())  # make scripts.*, utils.*, data_provider.* importable

HORIZONS = [96, 192, 336, 720]
DATASETS = ['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2']

# Paper Table II (multivariate), SegRNN column -- read directly off
# docs/SegRNN_paper.pdf's rendered table (page 6), not the paper's own
# text extraction (its table content doesn't extract as plain text).
PAPER_TABLE2 = {
    'ETTh1': {'mse': {96: 0.351, 192: 0.392, 336: 0.423, 720: 0.466},
              'mae': {96: 0.392, 192: 0.414, 336: 0.433, 720: 0.472}},
    'ETTh2': {'mse': {96: 0.276, 192: 0.341, 336: 0.364, 720: 0.403},
              'mae': {96: 0.335, 192: 0.389, 336: 0.403, 720: 0.448}},
    'ETTm1': {'mse': {96: 0.293, 192: 0.328, 336: 0.357, 720: 0.410},
              'mae': {96: 0.354, 192: 0.378, 336: 0.399, 720: 0.430}},
    'ETTm2': {'mse': {96: 0.164, 192: 0.226, 336: 0.284, 720: 0.381},
              'mae': {96: 0.250, 192: 0.294, 336: 0.339, 720: 0.402}},
}

# Paper Table III (univariate), SegRNN column -- same source, page 7.
PAPER_TABLE3 = {
    'ETTh1': {'mse': {96: 0.053, 192: 0.067, 336: 0.079, 720: 0.078},
              'mae': {96: 0.180, 192: 0.208, 336: 0.225, 720: 0.224}},
    'ETTh2': {'mse': {96: 0.125, 192: 0.160, 336: 0.186, 720: 0.209},
              'mae': {96: 0.277, 192: 0.320, 336: 0.350, 720: 0.370}},
    'ETTm1': {'mse': {96: 0.026, 192: 0.040, 336: 0.054, 720: 0.070},
              'mae': {96: 0.122, 192: 0.154, 336: 0.179, 720: 0.202}},
    'ETTm2': {'mse': {96: 0.063, 192: 0.089, 336: 0.117, 720: 0.156},
              'mae': {96: 0.183, 192: 0.224, 336: 0.263, 720: 0.309}},
}

# Exact per-dataset hyperparameters from scripts/SegRNN/<dataset>.sh
# (revin=0 matches run_longExp.py's own default, which is what these runs
# already used -- stated explicitly here only so every flag is visible.)
MULTIVARIATE_CONFIG = {
    'ETTh1': dict(seg_len=24, enc_in=7, d_model=512, dropout=0.1, channel_id=1, batch_size=64,  learning_rate=0.0003, revin=0),
    'ETTh2': dict(seg_len=24, enc_in=7, d_model=512, dropout=0.5, channel_id=0, batch_size=64,  learning_rate=0.0003, revin=0),
    'ETTm1': dict(seg_len=48, enc_in=7, d_model=512, dropout=0.5, channel_id=1, batch_size=256, learning_rate=0.0003, revin=0),
    'ETTm2': dict(seg_len=48, enc_in=7, d_model=512, dropout=0.5, channel_id=0, batch_size=256, learning_rate=0.0003, revin=0),
}

# Exact hyperparameters from scripts/SegRNN/univariate/<dataset>.sh -- the
# per-dataset, per-horizon scripts. NOT scripts/SegRNN/univariate.sh, the
# flat single-file version, which disagrees with these on nearly every
# setting; see the Part 2 markdown above for why these are the ones used.
#
# Structure: 'default' applies to every horizon, with per-horizon integer
# keys overriding it where the script itself differs by horizon.
UNIVARIATE_CONFIG = {
    'ETTh1': {
        'default': dict(seg_len=48, enc_in=1, d_model=256, dropout=0.5, channel_id=0, batch_size=256, learning_rate=0.0005, revin=0),
        # H=336/720 switch to shorter segments, wider d_model, far less dropout, smaller batches
        336:       dict(seg_len=24, enc_in=1, d_model=512, dropout=0.1, channel_id=0, batch_size=64,  learning_rate=0.0003, revin=0),
        720:       dict(seg_len=24, enc_in=1, d_model=512, dropout=0.1, channel_id=0, batch_size=64,  learning_rate=0.0003, revin=0),
    },
    'ETTh2': {
        'default': dict(seg_len=24, enc_in=1, d_model=128, dropout=0.1, channel_id=0, batch_size=64,  learning_rate=0.0003, revin=0),
    },
    'ETTm1': {
        'default': dict(seg_len=48, enc_in=1, d_model=512, dropout=0.5, channel_id=0, batch_size=256, learning_rate=0.0003, revin=0),
        # the only run in any ETT script that turns RevIN on
        720:       dict(seg_len=48, enc_in=1, d_model=512, dropout=0.5, channel_id=0, batch_size=256, learning_rate=0.0003, revin=1),
    },
    'ETTm2': {
        'default': dict(seg_len=48, enc_in=1, d_model=512, dropout=0.5, channel_id=0, batch_size=256, learning_rate=0.0003, revin=0),
    },
}


def uni_cfg(dataset, pred_len):
    """Univariate hyperparameters for one (dataset, horizon), applying any
    per-horizon override on top of the dataset's default."""
    d = UNIVARIATE_CONFIG[dataset]
    return dict(d.get(pred_len, d['default']))


# First univariate attempt, run with scripts/SegRNN/univariate.sh's settings
# before the discrepancy above was noticed. Kept -- not deleted -- so Part 3
# can quantify how much of the original Table III gap was hyperparameters
# rather than anything about the paper. Same code, same data, same seed;
# only the hyperparameters differ.
TABLE3_FIRST_ATTEMPT = {
    'ETTh1': {'mse': {96: 0.0529, 192: 0.0675, 336: 0.0788, 720: 0.0816},
              'mae': {96: 0.1803, 192: 0.2076, 336: 0.2249, 720: 0.2287}},
    'ETTh2': {'mse': {96: 0.1243, 192: 0.1687, 336: 0.1981, 720: 0.2266},
              'mae': {96: 0.2764, 192: 0.3294, 336: 0.3604, 720: 0.3836}},
    'ETTm1': {'mse': {96: 0.0258, 192: 0.0412, 336: 0.0551, 720: 0.0798},
              'mae': {96: 0.1213, 192: 0.1570, 336: 0.1834, 720: 0.2213}},
    'ETTm2': {'mse': {96: 0.0611, 192: 0.0908, 336: 0.1172, 720: 0.1614},
              'mae': {96: 0.1804, 192: 0.2266, 336: 0.2629, 720: 0.3148}},
}

# Measured reconstruction results, script configuration, single seed.
# Recorded so Parts 3-4 can be read and re-run without re-running Parts 1-2.
# These are the numbers printed by the Part 1 and Part 2 summaries above.
RECON_TABLE2 = {
    'ETTh1': {'mse': {96: 0.3510, 192: 0.3925, 336: 0.4232, 720: 0.4656},
              'mae': {96: 0.3925, 192: 0.4142, 336: 0.4327, 720: 0.4719}},
    'ETTh2': {'mse': {96: 0.2772, 192: 0.3413, 336: 0.3635, 720: 0.4014},
              'mae': {96: 0.3355, 192: 0.3890, 336: 0.4017, 720: 0.4454}},
    'ETTm1': {'mse': {96: 0.2962, 192: 0.3310, 336: 0.3588, 720: 0.4098},
              'mae': {96: 0.3570, 192: 0.3802, 336: 0.3992, 720: 0.4300}},
    'ETTm2': {'mse': {96: 0.1632, 192: 0.2269, 336: 0.2838, 720: 0.3790},
              'mae': {96: 0.2504, 192: 0.2946, 336: 0.3384, 720: 0.4008}},
}

RECON_TABLE3 = {
    'ETTh1': {'mse': {96: 0.0529, 192: 0.0675, 336: 0.0790, 720: 0.0778},
              'mae': {96: 0.1803, 192: 0.2076, 336: 0.2253, 720: 0.2242}},
    'ETTh2': {'mse': {96: 0.1249, 192: 0.1599, 336: 0.1857, 720: 0.2095},
              'mae': {96: 0.2772, 192: 0.3201, 336: 0.3498, 720: 0.3697}},
    'ETTm1': {'mse': {96: 0.0259, 192: 0.0401, 336: 0.0539, 720: 0.0700},
              'mae': {96: 0.1222, 192: 0.1537, 336: 0.1800, 720: 0.2028}},
    'ETTm2': {'mse': {96: 0.0629, 192: 0.0888, 336: 0.1167, 720: 0.1648},
              'mae': {96: 0.1827, 192: 0.2241, 336: 0.2624, 720: 0.3217}},
}


def script_cfg(dataset, pred_len, features):
    """The configuration the released scripts specify for one cell."""
    return dict(MULTIVARIATE_CONFIG[dataset]) if features == 'M' else uni_cfg(dataset, pred_len)


def paper_cfg(dataset, pred_len, features):
    """The configuration the paper's own text specifies for one cell.

    Section V-A2 calls seg_len=48 and d_model=512 uniform, and explicitly
    defers dropout, batch size and learning rate to the repository -- so
    those three keep their script values, since the paper points at the
    scripts for them. channel_id follows the architecture section, where
    CP encoding is half of PE=concat(rp,cp) and is described as disabled
    only in the univariate setting. RevIN is never mentioned for ETT.
    """
    cfg = script_cfg(dataset, pred_len, features)
    cfg['seg_len'] = 48
    cfg['d_model'] = 512
    cfg['channel_id'] = 1 if features == 'M' else 0
    cfg['revin'] = 0
    return cfg


def cfg_diff(script, paper):
    """Which flags the two configurations disagree on, as {flag: (script, paper)}."""
    return {k: (script[k], paper[k]) for k in script if script[k] != paper[k]}


os.makedirs('results/figures', exist_ok=True)


def run_horizon(dataset, pred_len, features, seg_len, enc_in, d_model, dropout, channel_id,
                 batch_size, learning_rate, revin=0, model='SegRNN'):
    """Launch run_longExp.py for one (dataset, horizon, features) combination,
    stream its output live, and parse the final 'mse:X, mae:Y, ms/sample:Z'
    line it prints. Returns (mse, mae, ms_per_sample)."""
    model_id = f'{dataset}_720_{pred_len}_{features}'
    cmd = [
        'python', '-u', 'run_longExp.py',
        '--is_training', '1', '--model_id', model_id, '--model', model, '--data', dataset,
        '--root_path', './dataset/', '--data_path', f'{dataset}.csv',
        '--features', features, '--seq_len', '720', '--pred_len', str(pred_len),
        '--seg_len', str(seg_len), '--enc_in', str(enc_in), '--d_model', str(d_model),
        '--dropout', str(dropout), '--rnn_type', 'gru', '--dec_way', 'pmf',
        '--channel_id', str(channel_id), '--revin', str(revin),
        '--train_epochs', '30', '--patience', '5',
        '--itr', '1', '--batch_size', str(batch_size), '--learning_rate', str(learning_rate),
    ]

    print(f'\n{"="*70}\n{dataset}  H={pred_len}  features={features}  '
          f'seg_len={seg_len} d_model={d_model} dropout={dropout} '
          f'channel_id={channel_id} bs={batch_size} lr={learning_rate} revin={revin}\n{"="*70}')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in proc.stdout:
        print(line, end='')
        tail.append(line)
        if len(tail) > 5:
            tail.pop(0)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'{dataset} H={pred_len} ({features}) failed (exit {proc.returncode}) -- see output above')

    m = re.search(r'mse:([\d.]+), mae:([\d.]+), ms/sample:([\d.]+)', ''.join(tail))
    if not m:
        raise RuntimeError(f'Could not find mse/mae/ms-per-sample in output for {dataset} H={pred_len} ({features})')
    return float(m.group(1)), float(m.group(2)), float(m.group(3))


# dataviz-validated categorical palette, same as colab_runner.ipynb's own Paper/Reconstruction pair
COLORS = {'Paper': '#2a78d6', 'Reconstruction': '#008300'}
INK_PRIMARY, INK_SECONDARY, INK_MUTED = '#0b0b0b', '#52514e', '#898781'
GRIDLINE, BASELINE_AXIS, SURFACE = '#e1e0d9', '#c3c2b7', '#fcfcfb'


def plot_metric(metric_name, series, title, save_path=None):
    """series: list of (label, {horizon: value}), in display order."""
    n_series = len(series)
    x = np.arange(len(HORIZONS))
    group_width = 0.8
    bar_width = group_width / n_series

    fig, ax = plt.subplots(figsize=(8, 5), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)
    for i, (label, values) in enumerate(series):
        offsets = x - group_width / 2 + bar_width * (i + 0.5)
        heights = [values[h] for h in HORIZONS]
        bars = ax.bar(offsets, heights, width=bar_width * 0.9, color=COLORS[label],
                       label=label, edgecolor=SURFACE, linewidth=0.5)
        for bar, h in zip(bars, heights):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{h:.3f}',
                     ha='center', va='bottom', fontsize=7.5, color=INK_PRIMARY)

    ax.set_xticks(x)
    ax.set_xticklabels([f'H={h}' for h in HORIZONS], color=INK_SECONDARY)
    ax.set_ylabel(metric_name.upper(), color=INK_SECONDARY)
    ax.set_title(title, color=INK_PRIMARY, fontsize=12, loc='left')
    ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ('top', 'right', 'left'):
        ax.spines[spine].set_visible(False)
    ax.spines['bottom'].set_color(BASELINE_AXIS)
    ax.tick_params(axis='both', which='both', length=0, colors=INK_MUTED)
    ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(0, 1.16),
              ncol=n_series, fontsize=9, labelcolor=INK_SECONDARY)

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=200, facecolor=SURFACE)
        print(f'saved {save_path}')
    plt.show()


def make_table(metric_name, series):
    rows = []
    for h in HORIZONS:
        row = {'Horizon': h}
        for label, values in series:
            row[label] = round(values[h], 4)
        rows.append(row)
    print(f'\n{metric_name.upper()}')
    return pd.DataFrame(rows).set_index('Horizon')


def show_comparison(title, results, paper, save_prefix):
    """results: {horizon: (mse, mae, ms)}. paper: {'mse': {...}, 'mae': {...}}."""
    series_mse = [('Paper', paper['mse']), ('Reconstruction', {h: results[h][0] for h in HORIZONS})]
    series_mae = [('Paper', paper['mae']), ('Reconstruction', {h: results[h][1] for h in HORIZONS})]
    display(make_table('mse', series_mse))
    display(make_table('mae', series_mae))
    plot_metric('mse', series_mse, f'{title} -- MSE', save_path=f'results/figures/{save_prefix}_mse.png')
    plot_metric('mae', series_mae, f'{title} -- MAE', save_path=f'results/figures/{save_prefix}_mae.png')

## Part 1 — Table II: multivariate reconstruction

`--features M`, all 7 channels in and out, channel-identity (CP) encoding
enabled where the dataset's own script turns it on. Hyperparameters below
are copied exactly from `scripts/SegRNN/<dataset>.sh` — note they differ
per dataset (segment length, dropout, and whether `channel_id` is even
used), not a single universal setting:

| Dataset | seg_len | d_model | dropout | channel_id | batch_size | lr |
|---|---|---|---|---|---|---|
| ETTh1 | 24 | 512 | 0.1 | 1 | 64 | 3e-4 |
| ETTh2 | 24 | 512 | 0.5 | 0 | 64 | 3e-4 |
| ETTm1 | 48 | 512 | 0.5 | 1 | 256 | 3e-4 |
| ETTm2 | 48 | 512 | 0.5 | 0 | 256 | 3e-4 |

Each dataset is its own Part (1a-1d) so a disconnect only costs that one
dataset's 4 runs, not the whole table.

### Part 1a -- ETTh1 (multivariate)

In [ ]:
table2_ETTh1 = {}
cfg = MULTIVARIATE_CONFIG['ETTh1']
for h in HORIZONS:
    table2_ETTh1[h] = run_horizon('ETTh1', h, features='M', **cfg)

show_comparison('ETTh1 -- Table II (multivariate)', table2_ETTh1, PAPER_TABLE2['ETTh1'], 'table2_ETTh1')

### Part 1b -- ETTh2 (multivariate)

In [ ]:
table2_ETTh2 = {}
cfg = MULTIVARIATE_CONFIG['ETTh2']
for h in HORIZONS:
    table2_ETTh2[h] = run_horizon('ETTh2', h, features='M', **cfg)

show_comparison('ETTh2 -- Table II (multivariate)', table2_ETTh2, PAPER_TABLE2['ETTh2'], 'table2_ETTh2')

### Part 1c -- ETTm1 (multivariate)

In [ ]:
table2_ETTm1 = {}
cfg = MULTIVARIATE_CONFIG['ETTm1']
for h in HORIZONS:
    table2_ETTm1[h] = run_horizon('ETTm1', h, features='M', **cfg)

show_comparison('ETTm1 -- Table II (multivariate)', table2_ETTm1, PAPER_TABLE2['ETTm1'], 'table2_ETTm1')

### Part 1d -- ETTm2 (multivariate)

In [ ]:
table2_ETTm2 = {}
cfg = MULTIVARIATE_CONFIG['ETTm2']
for h in HORIZONS:
    table2_ETTm2[h] = run_horizon('ETTm2', h, features='M', **cfg)

show_comparison('ETTm2 -- Table II (multivariate)', table2_ETTm2, PAPER_TABLE2['ETTm2'], 'table2_ETTm2')

### Part 1 summary -- Table II, all four datasets

In [ ]:
TABLE2_RESULTS = {'ETTh1': table2_ETTh1, 'ETTh2': table2_ETTh2, 'ETTm1': table2_ETTm1, 'ETTm2': table2_ETTm2}

rows = []
for name, results in TABLE2_RESULTS.items():
    for h in HORIZONS:
        mse, mae, ms = results[h]
        p_mse, p_mae = PAPER_TABLE2[name]['mse'][h], PAPER_TABLE2[name]['mae'][h]
        rows.append({
            'Dataset': name, 'Horizon': h,
            'Paper MSE': p_mse, 'Recon MSE': round(mse, 4), 'Delta MSE %': round((mse / p_mse - 1) * 100, 2),
            'Paper MAE': p_mae, 'Recon MAE': round(mae, 4), 'Delta MAE %': round((mae / p_mae - 1) * 100, 2),
        })
table2_summary_df = pd.DataFrame(rows).set_index(['Dataset', 'Horizon'])
display(table2_summary_df)
table2_summary_df.to_csv('results/stage1_table2_summary.csv')
print('saved results/stage1_table2_summary.csv')

## Part 2 — Table III: univariate reconstruction

`--features S`, a single channel (the `OT` target) in and out,
`channel_id=0` for every dataset (per the paper: "the CP encoding module
in SegRNN is disabled for this setting", since there's only one channel
to identify).

**Which script's hyperparameters these are, and why it matters.** The repo
ships *two* conflicting sets of univariate settings:

- `scripts/SegRNN/univariate.sh` — one flat file, one setting per dataset
- `scripts/SegRNN/univariate/<dataset>.sh` — per-dataset files, some of
  which vary *by horizon*

They disagree on nearly every value. The per-horizon files are used here,
for two reasons. First, they are the more specific of the two: a file that
bothers to give H=336 and H=720 their own `seg_len`, `d_model`, and
`dropout` is more plausibly the one that produced a published table than a
file applying one setting to all four horizons. Second — and this is the
empirical argument — an earlier version of this notebook used
`univariate.sh`, and its results (kept in `TABLE3_FIRST_ATTEMPT`) missed
the paper by a mean of 3.45% MSE, with the misses concentrated in exactly
the cells where the two scripts disagree most. Part 3 checks that claim
against the numbers rather than assuming it.

| Dataset | Horizon | seg_len | d_model | dropout | batch | lr | revin |
|---|---|---|---|---|---|---|---|
| ETTh1 | 96, 192 | 48 | 256 | 0.5 | 256 | 5e-4 | 0 |
| ETTh1 | 336, 720 | 24 | 512 | 0.1 | 64 | 3e-4 | 0 |
| ETTh2 | all | 24 | 128 | 0.1 | 64 | 3e-4 | 0 |
| ETTm1 | 96, 192, 336 | 48 | 512 | 0.5 | 256 | 3e-4 | 0 |
| ETTm1 | 720 | 48 | 512 | 0.5 | 256 | 3e-4 | **1** |
| ETTm2 | all | 48 | 512 | 0.5 | 256 | 3e-4 | 0 |

That single `revin=1` cell is the only run in any ETT script, univariate
or multivariate, that enables RevIN — and it lands on ETTm1 H=720, which
was the worst-reproducing cell of the first attempt (+13.9% MSE). Worth
watching specifically.

Again split into Parts 2a-2d, one per dataset.

### Part 2a -- ETTh1 (univariate)

In [ ]:
table3_ETTh1 = {}
for h in HORIZONS:
    table3_ETTh1[h] = run_horizon('ETTh1', h, features='S', **uni_cfg('ETTh1', h))

show_comparison('ETTh1 -- Table III (univariate)', table3_ETTh1, PAPER_TABLE3['ETTh1'], 'table3_ETTh1')

### Part 2b -- ETTh2 (univariate)

In [ ]:
table3_ETTh2 = {}
for h in HORIZONS:
    table3_ETTh2[h] = run_horizon('ETTh2', h, features='S', **uni_cfg('ETTh2', h))

show_comparison('ETTh2 -- Table III (univariate)', table3_ETTh2, PAPER_TABLE3['ETTh2'], 'table3_ETTh2')

### Part 2c -- ETTm1 (univariate)

In [ ]:
table3_ETTm1 = {}
for h in HORIZONS:
    table3_ETTm1[h] = run_horizon('ETTm1', h, features='S', **uni_cfg('ETTm1', h))

show_comparison('ETTm1 -- Table III (univariate)', table3_ETTm1, PAPER_TABLE3['ETTm1'], 'table3_ETTm1')

### Part 2d -- ETTm2 (univariate)

In [ ]:
table3_ETTm2 = {}
for h in HORIZONS:
    table3_ETTm2[h] = run_horizon('ETTm2', h, features='S', **uni_cfg('ETTm2', h))

show_comparison('ETTm2 -- Table III (univariate)', table3_ETTm2, PAPER_TABLE3['ETTm2'], 'table3_ETTm2')

### Part 2 summary -- Table III, all four datasets

In [ ]:
TABLE3_RESULTS = {'ETTh1': table3_ETTh1, 'ETTh2': table3_ETTh2, 'ETTm1': table3_ETTm1, 'ETTm2': table3_ETTm2}

rows = []
for name, results in TABLE3_RESULTS.items():
    for h in HORIZONS:
        mse, mae, ms = results[h]
        p_mse, p_mae = PAPER_TABLE3[name]['mse'][h], PAPER_TABLE3[name]['mae'][h]
        rows.append({
            'Dataset': name, 'Horizon': h,
            'Paper MSE': p_mse, 'Recon MSE': round(mse, 4), 'Delta MSE %': round((mse / p_mse - 1) * 100, 2),
            'Paper MAE': p_mae, 'Recon MAE': round(mae, 4), 'Delta MAE %': round((mae / p_mae - 1) * 100, 2),
        })
table3_summary_df = pd.DataFrame(rows).set_index(['Dataset', 'Horizon'])
display(table3_summary_df)
table3_summary_df.to_csv('results/stage1_table3_summary.csv')
print('saved results/stage1_table3_summary.csv')

### Part 2e -- diagnostic: does ETTm2 H=720 want RevIN?

After correcting the univariate hyperparameters, 15 of 16 cells land
within +/-0.70% of the paper. **ETTm2 H=720 is the exception** (+5.65%
MSE, +4.12% MAE) -- and the only cell that got *worse* under the
corrected config, since the learning-rate change that fixed ETTm2's three
shorter horizons did nothing for this one.

**Why RevIN is the hypothesis.** `scripts/SegRNN/univariate/ettm1.sh`
sets `--revin 1` at H=720 and `--revin 0` at every other horizon -- the
only run in any ETT script, univariate or multivariate, that enables it.
That single flag took ETTm1 H=720 from +13.94% to -0.01%. ETTm2 is the
same 15-minute frequency, same `seq_len`, `seg_len` and `d_model`, and
its own script is a flat `for pred_len in 96 192 336 720` loop with no
per-horizon handling at all -- the shape of a script that lost a special
case rather than one that never had one.

**Stating the methodological risk plainly.** Trying `revin=1` *because it
would make the number match* is fitting to the target, and that would be
a real problem if the result were reported selectively. Two things keep
this honest. First, the justification is the ETTm1 precedent, which
exists independently of whether this run helps. Second, the outcome is
reported either way, and this cell **deliberately does not overwrite**
`table3_ETTm2` -- the Part 2 summary above keeps reporting what
`ettm2.sh` actually specifies (`revin=0`), so the headline Table III
reconstruction stays faithful to the released script no matter what comes
out of this diagnostic.

In [ ]:
# Diagnostic only. Deliberately does NOT write back into table3_ETTm2, so the
# Part 2 summary keeps reporting the config the released script specifies.
ETTM2_720_REVIN0 = table3_ETTm2[720] if 'table3_ETTm2' in globals() else (0.1648, 0.3217, float('nan'))

cfg = uni_cfg('ETTm2', 720)
cfg['revin'] = 1
ettm2_720_revin1 = run_horizon('ETTm2', 720, features='S', **cfg)

p_mse, p_mae = PAPER_TABLE3['ETTm2']['mse'][720], PAPER_TABLE3['ETTm2']['mae'][720]
rows = []
for label, res in [('revin=0 (as scripted)', ETTM2_720_REVIN0),
                   ('revin=1 (diagnostic)', ettm2_720_revin1)]:
    mse, mae = res[0], res[1]
    rows.append({'Variant': label,
                 'MSE': round(mse, 4), 'Delta MSE %': round((mse / p_mse - 1) * 100, 2),
                 'MAE': round(mae, 4), 'Delta MAE %': round((mae / p_mae - 1) * 100, 2)})

print(f'ETTm2 H=720 univariate -- paper: MSE {p_mse}, MAE {p_mae}')
display(pd.DataFrame(rows).set_index('Variant'))

## Part 3 — Analysis

Written after Parts 1 and 2 produced real numbers, per the plan this
notebook was built to. Every figure below comes from the tables printed
above.

### 3.1 Did the reconstruction succeed?

Yes, on both tables.

| | Table II (multivariate) | Table III (univariate) |
|---|---|---|
| Mean absolute Δ, MSE | 0.34% | 0.56% (**0.22%** excluding ETTm2 H=720) |
| Mean absolute Δ, MAE | 0.23% | 0.42% (**0.17%** excluding that cell) |
| Mean *signed* Δ, MSE | +0.12% | +0.29% (**−0.07%** excluding it) |
| Cells within ±1.1% | 16 of 16 | 15 of 16 |

The signed-versus-absolute comparison is the part that matters. Errors
that scatter symmetrically around zero are run-to-run noise; a mean
signed error close to zero means the reconstruction is not systematically
better or worse than the paper, just imprecise at the fourth decimal.
That is what reproduction is supposed to look like. Table II additionally
shows no horizon trend (+0.26 / +0.38 / +0.09 / −0.27% at H=96/192/336/720),
so nothing degrades as the forecast gets longer.

**31 of 32 reported cells reproduce.** One does not; Part 2e
investigated it and failed to explain it, which 3.5 records.

### 3.2 The control that makes the rest of this trustworthy

Part 2 was run twice, because the first attempt used the wrong
hyperparameter source (see 3.3). ETTh1 H=96 and H=192 are the only two
cells where the two candidate scripts happen to **agree**, so their
configuration was identical across both attempts. Their results:

| Cell | First attempt | Re-run |
|---|---|---|
| ETTh1 H=96 | −0.22% | −0.22% |
| ETTh1 H=192 | +0.70% | +0.70% |

Unchanged to two decimals. This is a genuine control, and it establishes
something the rest of the analysis depends on: **the pipeline is
seed-deterministic**, so every other change between the two attempts is
attributable to hyperparameters and not to run-to-run variance. Without
it, "the config was wrong" and "we got a bad draw" would be
indistinguishable.

### 3.3 The accidental experiment: how hyperparameter-sensitive is SegRNN?

The repository ships **two contradictory sets of univariate
hyperparameters**: `scripts/SegRNN/univariate.sh`, a single flat file with
one setting per dataset, and `scripts/SegRNN/univariate/<dataset>.sh`,
per-dataset files that in two cases vary *by horizon*. Nothing marks
which is authoritative. The first attempt used the flat file; the re-run
used the per-horizon files. Because of the control in 3.2, the difference
between them is a clean measurement:

| Dataset | H | Flat script | Per-horizon script | Config difference |
|---|---|---|---|---|
| ETTh1 | 720 | +4.57% | −0.31% | seg_len, d_model, dropout, batch, lr |
| ETTh2 | 192 | +5.41% | −0.09% | seg_len, d_model, dropout, batch, lr |
| ETTh2 | 336 | +6.51% | −0.17% | " |
| ETTh2 | 720 | +8.43% | +0.22% | " |
| ETTm1 | 720 | **+13.94%** | **−0.01%** | lr, **and `--revin 1`** |
| ETTm2 | 720 | +3.46% | +5.65% | lr only — *worsened* |

Aggregate: mean absolute MSE error fell from **3.45% to 0.56%**, and the
systematic bias from **+2.81% to +0.29%**.

The ETTm1 H=720 row is the striking one. That cell went from the worst
in the table to the best-matching in it, and the decisive difference is a
single boolean flag — RevIN, SegRNN's reversible instance normalization.
Turning it on is worth roughly 14% MSE at that one cell.

**What this says about the model.** SegRNN's univariate results are far
more hyperparameter-sensitive than its multivariate ones. Table II
reproduced on the first attempt, with one configuration per dataset
applied uniformly across all four horizons. Table III did not: it needed
per-horizon `seg_len` and `d_model` for ETTh1, a completely different
architecture size for ETTh2, and a normalization flag for ETTm1 H=720.

A plausible mechanism, offered as a hypothesis rather than a tested
finding: SegRNN is channel-independent, so in the multivariate setting a
batch of 64 windows carries 7× as many channel-instances through each
gradient step, and the reported metric is averaged over 7 channels. Both
effects damp sensitivity — more effective data per update, and averaging
that suppresses per-channel idiosyncrasy in the score. The univariate
setting has neither. Testing this properly would mean re-running the
*multivariate* configuration with deliberately mismatched hyperparameters
to see whether it degrades as sharply; that was not done here, so this
remains a mechanism proposal.

### 3.4 A reproducibility observation, and where it leads

Table III's numbers depend on per-horizon tuning that appears nowhere in
the paper's text — only in the released scripts, and there only in the
less obvious of two contradictory copies. A reader working from the PDF
alone would land close to this project's first attempt: roughly 3.5% off,
with the gap widening at longer horizons.

**It is broader than Table III, though.** The paper's Section V-A2
describes a "uniform configuration" of segment length 48 and hidden size
512. The released scripts contradict that on both counts — `seg_len=24`
for ETTh1 and ETTh2 multivariate, `d_model` of 256 or 128 in several
univariate runs — and additionally disable the CP half of
`PE=concat(rp,cp)` (`channel_id=0`) in two of the four multivariate ETT
runs, which the text never mentions. Most of the stated configuration
does hold exactly: look-back 720, one GRU layer, 30 epochs, patience 5,
and `--lradj type3`, which is literally the paper's "decay of 0.8 after
the initial three epochs." And the paper explicitly defers dropout, batch
size and learning rate to the repository, so drawing those from the
scripts is what it instructs.

**The expectation, stated before running it:** Parts 1-2 reproduce the
published tables using the script configuration, so the script
configuration is presumably what produced them, and re-running the
disagreeing cells under the paper's stated configuration should miss.
**Part 4 tests exactly that**, and if the paper config matches just as
well, this expectation was wrong and should be recorded as wrong.

That the reconstruction already succeeds at `seg_len=24` for ETTh1 (0.01%
to 0.14% from the published values) is suggestive but not conclusive on
its own: it shows `w=24` is consistent with the published number, not
that `w=48` would be inconsistent with it. Only Part 4 can settle that,
which is why it exists. Note that the paper's own Fig. 6 shows forecast
error varying materially with `w` — so a 2× change in `w` making almost
no difference would itself be a notable result.

This is worth stating without accusation. A stale convenience script
shipped alongside the real one, and a configuration sentence written
before the final per-dataset tuning, are both ordinary in released
research, and the authors did release everything needed to reproduce the
result. But it does mean the honest description of these tables is
"reproducible from the repository," not "reproducible from the paper."

### 3.5 The one unreproduced cell

**ETTm2 H=720: +5.64% MSE, +4.11% MAE.** Part 2e tested the RevIN
hypothesis from 3.3 — that this cell wants `--revin 1` the way ETTm1
H=720 does — and it is **wrong**:

| Variant | MSE | Δ MSE | MAE | Δ MAE |
|---|---|---|---|---|
| `revin=0` (as scripted) | 0.1648 | +5.64% | 0.3217 | +4.11% |
| `revin=1` (diagnostic) | 0.1686 | **+8.10%** | 0.3263 | **+5.60%** |

RevIN makes this cell worse on both metrics, not better. Taking the
conclusion that was committed to in advance: **ETTm2 H=720 stands as an
honestly unreproduced cell, 1 of 32.** That does not change the verdict
in 3.1.

Three things are known about it, none of which amount to an explanation:

- It is not run-to-run variance, by the control in 3.2.
- Learning rate moves it in the *opposite* direction to every other cell.
  The flat script's `lr=1e-4` gave +3.46%; the per-horizon script's
  `lr=3e-4` gives +5.64%. This is the only cell in the entire table where
  the flat script produced the better result.
- RevIN, the one mechanism with a precedent at this exact horizon, makes
  it worse.
- **It is not the dataset file.** The same `ETTm2.csv` produces a correct
  multivariate H=720 result in Table II (-0.53% MSE), and correct
  univariate results at H=96/192/336 (-0.18/-0.21/-0.25%). A wrong or
  differently-versioned data file is a property of the file, so it would
  move every horizon and both tables; seven of ETTm2’s eight reported
  cells reproduce. The multivariate H=720 run is the sharpest form of
  this control, since it reads the same rows, the same split and the
  same `OT` column at the same horizon, differing only in `--features`.

**Why the investigation stops here, deliberately.** Three hyperparameter
settings have now been tried against a target value that is known in
advance. Continuing — sweeping the learning rate lower, extending
`train_epochs`, varying `seg_len` — would eventually produce a
configuration whose output is close to 0.156, and that number would carry
no evidential weight, because a search that terminates when it hits a
known target reports the target rather than a measurement. The line worth
holding: characterizing *uncertainty* is legitimate, searching for a
*match* is not.

If this cell is worth pursuing further, the legitimate move is the second
kind. This project's own protocol (`CLAUDE.md`) specifies three seeds
(2021, 2022, 2023) reported as mean ± std; Stage 1 was run single-seed
throughout, here and everywhere else. Running ETTm2 H=720 across those
three seeds would establish whether 5.6% falls inside the cell's own seed
spread — a variance measurement made without reference to whether the
answer is convenient. It would also be worth doing for its own sake: a
single-seed reconstruction is the one methodological gap this notebook
has relative to the standard the project set for itself.

### 3.6 What carries into Stage 2

Two things, both practical.

**The Stage 2 baseline is trustworthy.** Every Stage 2 improvement in
this project is measured against the ETTh1 multivariate reconstruction
(MSE 0.3510 / 0.3925 / 0.4232 / 0.4656 at H=96/192/336/720), which
matches the paper to between 0.01% and 0.14%. Improvements measured
against it are therefore improvements over the paper's actual reported
model, not over a weakened copy of it — which is exactly the property a
Stage 1 reconstruction has to establish before Stage 2 means anything.

**A caution about Stage 2's negative results.** Section 3.3 shows a
single flag moving one cell by 14%. Every Stage 2 strand in this project
was tested at one fixed configuration, inherited from the reconstruction.
That is the right design for isolating each strand's effect — but it
means a strand recorded as "regressed" was shown to regress *at that
configuration*, which is not the same as being a bad idea. This does not
undermine the negative results, and they should still be reported as
found; it does mean they should be phrased as "did not help here" rather
than "does not work."

## Part 4 — The paper's stated configuration vs. the released scripts

Parts 1 and 2 reconstruct the published tables using the hyperparameters
in the repository's own scripts, and they succeed: 31 of 32 cells within
about 1%. But the scripts are not what the *paper* says. Section V-A2
states:

> "The **uniform configuration** of SegRNN consists of a look-back of
> 720, a **segment length of 48**, a single GRU layer, a **hidden size of
> 512**, 30 training epochs, a learning rate decay of 0.8 after the
> initial three epochs, and early stopping with a patience of 5. The
> dropout rate, batch size, and learning rate vary based on the scale of
> the data. The complete set of parameters used for SegRNN across
> different datasets is available in our open-source repository."

Most of that checks out exactly: `seq_len=720`, one GRU layer, 30 epochs,
`patience=5`, and `--lradj type3`, which is literally
`lr * 0.8^(epoch-3)` for `epoch >= 3` and is the default no script
overrides. The paper also explicitly hands dropout, batch size and
learning rate to the repository, so taking those from the scripts is what
the paper instructs, not a shortcut.

**Three things do not check out.**

| Flag | Paper says | Scripts use | Where |
|---|---|---|---|
| `seg_len` | 48, uniform | **24** | ETTh1/ETTh2 multivariate; ETTh1 uni H=336/720; ETTh2 uni |
| `d_model` | 512, uniform | **256 / 128** | ETTh1 uni H=96/192; ETTh2 uni |
| `channel_id` | CP is half of `PE=concat(rp,cp)`; disabled only for univariate | **0** | ETTh2/ETTm2 **multivariate** — CP off |
| `revin` | never mentioned for ETT | **1** | ETTm1 uni H=720 |

The `channel_id` row is the one the paper's text does not acknowledge at
all: two of the four multivariate ETT runs disable a component the
architecture section presents as standard.

**What this Part does.** It re-runs only the cells where the two
configurations disagree — about 21 of the 32 — and reports the published
number, the script-config reconstruction, and the paper-config
reconstruction side by side. ETTm1 multivariate and ETTm2 univariate are
already fully paper-conformant and need no run at all.

**A design decision worth stating.** Only the flags the paper actually
specifies are changed. Dropout, batch size and learning rate keep their
script values even where they vary by horizon, because the paper defers
them to the repository rather than fixing them. This isolates the
disagreement instead of confounding it with a second set of changes.

**What the outcome would mean.** If the paper-config runs miss the
published tables while the script-config runs hit them, that is direct
evidence the published numbers were produced by the scripts and the
paper's "uniform configuration" sentence does not describe them — which
would make Table II and Table III reproducible from the repository but
not from the paper. If the two configurations land in the same place,
then `seg_len` and `d_model` simply do not matter much here, which would
be a surprise given the paper's own Fig. 6 shows forecast error varying
materially with `w`.

In [ ]:
# What actually differs. Prints the plan without running anything.
PART4_CELLS = []
for features, cfgs in (('M', 'multivariate'), ('S', 'univariate')):
    for ds in DATASETS:
        for h in HORIZONS:
            sc, pc = script_cfg(ds, h, features), paper_cfg(ds, h, features)
            d = cfg_diff(sc, pc)
            if d:
                PART4_CELLS.append((ds, features, h, d))

rows = [{'Dataset': ds, 'Features': f, 'Horizon': h,
         'Changed': ', '.join(f'{k}: {a}->{b}' for k, (a, b) in d.items())}
        for ds, f, h, d in PART4_CELLS]
print(f'{len(PART4_CELLS)} of 32 cells differ between the two configurations '
      f'({16 - sum(1 for c in PART4_CELLS if c[1] == "M")} multivariate and '
      f'{16 - sum(1 for c in PART4_CELLS if c[1] == "S")} univariate cells already agree).')
display(pd.DataFrame(rows).set_index(['Dataset', 'Features', 'Horizon']))

In [ ]:
# Part 4a -- multivariate cells that differ
table4_results = {}
for ds, features, h, _ in PART4_CELLS:
    if features != 'M':
        continue
    table4_results[(ds, features, h)] = run_horizon(ds, h, features=features, **paper_cfg(ds, h, features))

print(f'done: {sum(1 for k in table4_results if k[1] == "M")} multivariate paper-config runs')

In [ ]:
# Part 4b -- univariate cells that differ
for ds, features, h, _ in PART4_CELLS:
    if features != 'S':
        continue
    table4_results[(ds, features, h)] = run_horizon(ds, h, features=features, **paper_cfg(ds, h, features))

print(f'done: {sum(1 for k in table4_results if k[1] == "S")} univariate paper-config runs')

In [ ]:
# Part 4 summary -- published vs script config vs paper config
PAPER = {'M': PAPER_TABLE2, 'S': PAPER_TABLE3}
RECON = {'M': RECON_TABLE2, 'S': RECON_TABLE3}

rows = []
for ds, features, h, d in PART4_CELLS:
    if (ds, features, h) not in table4_results:
        continue
    pub_mse, pub_mae = PAPER[features][ds]['mse'][h], PAPER[features][ds]['mae'][h]
    sc_mse, sc_mae = RECON[features][ds]['mse'][h], RECON[features][ds]['mae'][h]
    pc_mse, pc_mae = table4_results[(ds, features, h)][:2]
    rows.append({
        'Dataset': ds, 'Feat': features, 'Horizon': h,
        'Changed': ', '.join(d.keys()),
        'Published MSE': pub_mse,
        'Script d%': round((sc_mse / pub_mse - 1) * 100, 2),
        'Paper-cfg d%': round((pc_mse / pub_mse - 1) * 100, 2),
        'Published MAE': pub_mae,
        'Script d% ': round((sc_mae / pub_mae - 1) * 100, 2),
        'Paper-cfg d% ': round((pc_mae / pub_mae - 1) * 100, 2),
    })

table4_df = pd.DataFrame(rows).set_index(['Dataset', 'Feat', 'Horizon'])
display(table4_df)
table4_df.to_csv('results/stage1_table4_paper_vs_script.csv')
print('saved results/stage1_table4_paper_vs_script.csv')

print('\nMean absolute deviation from the published tables, over these cells only:')
print(f"  script config : MSE {table4_df['Script d%'].abs().mean():.2f}%   "
      f"MAE {table4_df['Script d% '].abs().mean():.2f}%")
print(f"  paper config  : MSE {table4_df['Paper-cfg d%'].abs().mean():.2f}%   "
      f"MAE {table4_df['Paper-cfg d% '].abs().mean():.2f}%")

### Part 4 — reading the result

*To be completed once Part 4 has run.* The comparison to make is the two
mean-deviation lines printed above: whichever configuration sits closer
to the published tables is the one that produced them. Section 3.4 of
Part 3 states the expectation in advance — that the script config wins —
so if the paper config turns out to match just as well, that expectation
was wrong and should be recorded as such rather than quietly dropped.

## Optional — save results back into the repo

Appends this notebook's rows to `results/runs.csv`. Commit/push left
commented out on purpose — review `git status`/`git diff` first.

In [ ]:
RUNS_CSV_HEADER = ['run_id','timestamp','model','dataset','horizon','seq_len','seg_len',
                    'd_model','seed','flags','mse','mae','mase','epoch_time_s','params',
                    'peak_mem_mb','notes']
ts = datetime.datetime.now().isoformat(timespec='seconds')
rows = []


def _flags(features, cfg):
    return (f"features={features};dropout={cfg['dropout']};channel_id={cfg['channel_id']};"
            f"revin={cfg['revin']};bs={cfg['batch_size']};lr={cfg['learning_rate']}")


# Part 1 and Part 2 are independently runnable, so either half may be absent
# from the kernel -- log whichever ones actually ran rather than failing.
for name, results in globals().get('TABLE2_RESULTS', {}).items():
    cfg = MULTIVARIATE_CONFIG[name]
    for h, (mse, mae, ms) in results.items():
        rows.append([f'SegRNN_{name}_{h}_M_{ts}', ts, 'SegRNN', name, h, 720, cfg['seg_len'], cfg['d_model'], 2024,
                     _flags('M', cfg),
                     mse, mae, '', '', '', '', 'Stage 1 Table II reconstruction'])

# Univariate settings vary by horizon (seg_len and d_model included), so the
# config has to be looked up per horizon, not per dataset.
for name, results in globals().get('TABLE3_RESULTS', {}).items():
    for h, (mse, mae, ms) in results.items():
        cfg = uni_cfg(name, h)
        rows.append([f'SegRNN_{name}_{h}_S_{ts}', ts, 'SegRNN', name, h, 720, cfg['seg_len'], cfg['d_model'], 2024,
                     _flags('S', cfg),
                     mse, mae, '', '', '', '', 'Stage 1 Table III reconstruction'])

if not rows:
    raise RuntimeError('Neither TABLE2_RESULTS nor TABLE3_RESULTS is defined -- run Part 1 and/or Part 2 first.')

existing = pd.read_csv('results/runs.csv') if os.path.exists('results/runs.csv') else pd.DataFrame(columns=RUNS_CSV_HEADER)
new_df = pd.DataFrame(rows, columns=RUNS_CSV_HEADER)
combined = pd.concat([existing, new_df], ignore_index=True)
combined.to_csv('results/runs.csv', index=False)
print(f'appended {len(rows)} rows to results/runs.csv (total {len(combined)})')

!git add -A results/
!git status
# review the diff above, then when ready:
# !git commit -m "Stage 1: reconstruct Table II and Table III for ETTh1/ETTh2/ETTm1/ETTm2"
# !git push origin main